# 🗺️ Exercícios — Busca Heurística (Greedy, A*)

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Implemente algoritmos de busca informada e compare com buscas cegas. Explore o impacto de diferentes heurísticas.


## 1. Heurística de Manhattan e Euclidiana

In [ ]:
import math

def heuristica_manhattan(pos, destino):
    """Distância de Manhattan: ideal para grids sem movimentos diagonais."""
    return abs(pos[0]-destino[0]) + abs(pos[1]-destino[1])

def heuristica_euclidiana(pos, destino):
    """Distância Euclidiana: boa para espaços contínuos."""
    return math.sqrt((pos[0]-destino[0])**2 + (pos[1]-destino[1])**2)

destino = (5, 5)
pontos = [(0,0), (1,3), (3,1), (4,4), (5,5)]
print(f"{'Ponto':^10} | {'Manhattan':^12} | {'Euclidiana':^12}")
print("-" * 40)
for p in pontos:
    print(f"{str(p):^10} | {heuristica_manhattan(p,destino):^12.2f} | {heuristica_euclidiana(p,destino):^12.2f}")


## 2. Algoritmo A* em um Grid

In [ ]:
import heapq
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

labirinto = [
    [0,0,0,0,1,0,0,0,0,0],
    [0,1,1,0,1,0,1,1,1,0],
    [0,0,0,0,0,0,0,0,1,0],
    [0,1,1,1,1,1,1,0,1,0],
    [0,0,0,0,0,0,1,0,0,0],
    [1,1,0,1,1,0,1,1,1,0],
    [0,0,0,0,1,0,0,0,0,0],
    [0,1,1,0,1,1,1,1,0,1],
    [0,0,0,0,0,0,0,1,0,0],
    [0,0,0,1,1,1,0,0,0,0],
]
INICIO, FIM = (0,0), (9,9)
R, C = len(labirinto), len(labirinto[0])

def a_estrela(grid, inicio, fim, heuristica):
    heap = [(0+heuristica(inicio,fim), 0, inicio, [inicio])]
    visitados = {}
    while heap:
        f, g, pos, caminho = heapq.heappop(heap)
        if pos in visitados: continue
        visitados[pos] = g
        if pos == fim:
            return caminho, visitados
        r, c = pos
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr,nc = r+dr, c+dc
            if 0<=nr<R and 0<=nc<C and grid[nr][nc]==0 and (nr,nc) not in visitados:
                ng = g + 1
                nf = ng + heuristica((nr,nc), fim)
                heapq.heappush(heap, (nf, ng, (nr,nc), caminho+[(nr,nc)]))
    return None, visitados

cam_man, vis_man = a_estrela(labirinto, INICIO, FIM, heuristica_manhattan)
cam_euc, vis_euc = a_estrela(labirinto, INICIO, FIM, heuristica_euclidiana)

def visualizar(cam, vis, titulo):
    grid = np.array(labirinto, dtype=float)
    fig, ax = plt.subplots(figsize=(7,7))
    ax.imshow(grid, cmap='binary', vmin=0, vmax=1)
    for (r,c) in vis: ax.add_patch(plt.Rectangle((c-.5,r-.5),1,1,color='lightblue',alpha=0.5))
    if cam:
        for (r,c) in cam: ax.add_patch(plt.Rectangle((c-.5,r-.5),1,1,color='green',alpha=0.7))
    ax.add_patch(plt.Rectangle((INICIO[1]-.5,INICIO[0]-.5),1,1,color='yellow'))
    ax.add_patch(plt.Rectangle((FIM[1]-.5,FIM[0]-.5),1,1,color='red'))
    ax.set_xticks(range(C)); ax.set_yticks(range(R)); ax.grid(True)
    passos = len(cam) if cam else 0
    ax.set_title(f'{titulo} | {passos} passos | {len(vis)} nós')
    plt.show()

visualizar(cam_man, vis_man, 'A* com Heurística Manhattan')
visualizar(cam_euc, vis_euc, 'A* com Heurística Euclidiana')

print(f"Manhattan: {len(cam_man) if cam_man else 'N/A'} passos, {len(vis_man)} nós explorados")
print(f"Euclidiana: {len(cam_euc) if cam_euc else 'N/A'} passos, {len(vis_euc)} nós explorados")


### 📝 Exercício 1

Implemente a **heurística zero** (h=0) e passe para o A*. O que acontece? Qual busca conhecida o A* imita quando h=0?

In [ ]:
def heuristica_zero(pos, destino):
    return 0  # heurística nula

cam_zero, vis_zero = a_estrela(labirinto, INICIO, FIM, heuristica_zero)
cam_man2, vis_man2 = a_estrela(labirinto, INICIO, FIM, heuristica_manhattan)

print(f"h=0  (= UCS/BFS):  {len(cam_zero) if cam_zero else 'N/A'} passos, {len(vis_zero)} nós")
print(f"h=Manhattan:       {len(cam_man2) if cam_man2 else 'N/A'} passos, {len(vis_man2)} nós")
print(f"\nCom h=0, A* comporta-se como: UCS / BFS (busca cega de custo uniforme)")


## 3. Greedy Best-First Search

Diferente do A*, a busca gulosa usa **apenas a heurística** (ignora o custo já percorrido).

In [ ]:
def greedy_bfs(grid, inicio, fim, heuristica):
    """Greedy Best-First Search — usa apenas h(n)."""
    heap = [(heuristica(inicio,fim), inicio, [inicio])]
    visitados = set()
    while heap:
        h, pos, caminho = heapq.heappop(heap)
        if pos in visitados: continue
        visitados.add(pos)
        if pos == fim:
            return caminho, visitados
        r, c = pos
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr,nc = r+dr, c+dc
            if 0<=nr<R and 0<=nc<C and grid[nr][nc]==0 and (nr,nc) not in visitados:
                heapq.heappush(heap, (heuristica((nr,nc),fim), (nr,nc), caminho+[(nr,nc)]))
    return None, visitados

cam_greedy, vis_greedy = greedy_bfs(labirinto, INICIO, FIM, heuristica_manhattan)

print("Comparação A* vs Greedy:")
print(f"  A* (Manhattan): {len(cam_man):3d} passos, {len(vis_man):3d} nós explorados")
print(f"  Greedy:         {len(cam_greedy) if cam_greedy else 'N/A':3} passos, {len(vis_greedy):3d} nós explorados")
print()
print("A* garante o caminho ótimo; Greedy é mais rápido mas pode não ser ótimo.")


### 📝 Exercício 2

Crie um labirinto **modificado** onde o Greedy encontra um caminho **não-ótimo** enquanto o A* encontra o ótimo. Mostre os dois resultados.

In [ ]:
# ✏️ Crie um labirinto onde Greedy falha em encontrar o caminho ótimo
labirinto_armadilha = [
    [0, 0, 0, 0, 0],
    [1, 1, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
]
# TODO: defina INICIO2 e FIM2, execute A* e Greedy, compare
INICIO2, FIM2 = (0,0), (4,4)
R2, C2 = 5, 5


## 4. Exercício Final — Puzzle 8

O puzzle 8 (3×3) é um problema clássico. Resolva com A* usando heurística de Manhattan.

In [ ]:
from collections import namedtuple

Estado = namedtuple('Estado', ['tabuleiro', 'g', 'caminho'])

def manhattan_puzzle(tabuleiro, objetivo):
    """Soma das distâncias de Manhattan de cada peça."""
    total = 0
    for i in range(9):
        val = tabuleiro[i]
        if val == 0: continue
        pos_obj = objetivo.index(val)
        total += abs(i//3 - pos_obj//3) + abs(i%3 - pos_obj%3)
    return total

def a_estrela_puzzle(inicio, objetivo):
    objetivo_t = tuple(objetivo)
    h0 = manhattan_puzzle(inicio, objetivo_t)
    heap = [(h0+0, 0, inicio, [])]
    visitados = set()
    while heap:
        f, g, tab, cam = heapq.heappop(heap)
        if tab in visitados: continue
        visitados.add(tab)
        if tab == objetivo_t:
            return cam + [tab], len(visitados)
        pos0 = tab.index(0)
        r, c = pos0//3, pos0%3
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr,nc = r+dr, c+dc
            if 0<=nr<3 and 0<=nc<3:
                npos = nr*3+nc
                novo = list(tab); novo[pos0],novo[npos] = novo[npos],novo[pos0]
                novo_t = tuple(novo)
                if novo_t not in visitados:
                    ng = g+1
                    heapq.heappush(heap,(ng+manhattan_puzzle(novo_t,objetivo_t),ng,novo_t,cam+[tab]))
    return None, len(visitados)

inicio  = (1,2,5, 3,4,0, 6,7,8)
objetivo = (1,2,5, 3,4,8, 6,7,0)
solucao, nos = a_estrela_puzzle(inicio, objetivo)
print(f"Puzzle 8 resolvido em {len(solucao)-1} movimentos, {nos} nós explorados")
print("Estados finais:")
for s in solucao:
    for i in range(0,9,3): print(" ", s[i:i+3])
    print()
